# 01 — RL Basics: DQN on CartPole

**Module 4 · Week 7 · Phạm Ngọc Khánh**

This notebook covers:
- Gymnasium environment interaction loop
- Replay buffer
- Q-network and target network
- ε-greedy action selection with decay
- DQN training loop
- Reward curve plotting

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Gymnasium version: {gym.__version__}')

## 1. Explore the Environment

In [ ]:
env = gym.make('CartPole-v1')
obs, info = env.reset(seed=42)

print(f'Observation space: {env.observation_space}')   # Box(4,)
print(f'Action space:      {env.action_space}')        # Discrete(2)
print(f'Sample observation: {obs}')

# Run one random episode
total_reward = 0
obs, _ = env.reset()
for step in range(200):
    action = env.action_space.sample()  # random action
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f'Random policy — episode reward: {total_reward}')
env.close()

## 2. Replay Buffer

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.long, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(np.array(next_states), dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)

## 3. Q-Network

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim: int = 4, action_dim: int = 2, hidden: int = 128):
        super().__init__()
        # --- TODO: Define MLP ---
        # Input: state_dim
        # Hidden: 2 layers of `hidden` units with ReLU
        # Output: action_dim (Q-value for each action)
        pass

    def forward(self, x):
        pass

policy_net = QNetwork().to(device)
target_net = QNetwork().to(device)
target_net.load_state_dict(policy_net.state_dict())  # same initial weights
target_net.eval()

print(f'Q-network parameters: {sum(p.numel() for p in policy_net.parameters()):,}')

## 4. DQN Training

In [ ]:
# Hyperparameters
BUFFER_SIZE = 10_000
BATCH_SIZE = 64
GAMMA = 0.99          # discount factor
LR = 1e-3
EPS_START = 1.0       # initial exploration rate
EPS_END = 0.05        # final exploration rate
EPS_DECAY = 500       # episodes over which eps decays
TARGET_UPDATE = 100   # update target network every N steps
NUM_EPISODES = 600

replay_buffer = ReplayBuffer(BUFFER_SIZE)
optimizer = torch.optim.Adam(policy_net.parameters(), lr=LR)

episode_rewards = []
eps = EPS_START
total_steps = 0

env = gym.make('CartPole-v1')

for episode in range(NUM_EPISODES):
    state, _ = env.reset()
    episode_reward = 0

    for step in range(500):  # max steps per episode
        # --- TODO: ε-greedy action selection ---
        # With probability eps: random action
        # Otherwise: argmax of Q-values from policy_net
        action = None  # replace

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        episode_reward += reward

        replay_buffer.push(state, action, reward, next_state, done)
        state = next_state
        total_steps += 1

        # --- TODO: Training step (only when buffer has enough samples) ---
        if len(replay_buffer) >= BATCH_SIZE:
            # 1. Sample mini-batch from replay buffer
            # 2. Compute target Q-values using target_net
            #    target = reward + gamma * max(Q_target(next_state)) * (1 - done)
            # 3. Compute current Q-values from policy_net
            # 4. Loss = MSE(Q_current, target)
            # 5. Backprop and clip gradients (max_norm=1.0)
            pass

        # --- TODO: Update target network every TARGET_UPDATE steps ---

        if done:
            break

    # Decay epsilon
    eps = max(EPS_END, EPS_START - episode / EPS_DECAY)
    episode_rewards.append(episode_reward)

    if (episode + 1) % 50 == 0:
        avg = np.mean(episode_rewards[-50:])
        print(f'Episode {episode+1}/{NUM_EPISODES}  Avg(50): {avg:.1f}  ε: {eps:.3f}')

env.close()

## 5. Plot Reward Curve

In [ ]:
def moving_average(data, window=20):
    return np.convolve(data, np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, alpha=0.3, label='Episode reward')
plt.plot(moving_average(episode_rewards), label='Moving average (20)')
plt.axhline(195, color='green', linestyle='--', label='Solved threshold (195)')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.title('DQN on CartPole-v1')
plt.legend()
plt.grid(True)
plt.show()

print(f'Max reward: {max(episode_rewards)}')
print(f'Final 50-episode average: {np.mean(episode_rewards[-50:]):.1f}')

## 6. Evaluate Greedy Policy

In [ ]:
# --- TODO: Run 10 evaluation episodes with epsilon = 0 (greedy) ---
# Print reward for each episode
